# SOL-Hi — OOD Holdout vs Random Shuffle Protocol Comparison

This notebook compares two inner hyperparameter-selection protocols on the SOL-Hi task:

1. **OOD holdout**: the validation subset is chemically separated from the inner training subset.
2. **Random shuffle**: the validation subset is obtained by randomly splitting the outer training set.

The main experimental question is:

> Does in-distribution hyperparameter selection produce higher internal validation scores without improving final out-of-distribution test performance?

The analysis focuses on:

- inner validation PR-AUC;
- inner train PR-AUC;
- outer train PR-AUC after refitting;
- final OOD test PR-AUC;
- inner-to-test generalization gap;
- train-to-test generalization gap;
- selected hyperparameters across folds.

The models considered are:

- Decision Tree;
- Logistic Regression;
- Linear SVM.

The fingerprints considered are:

- ECFP4;
- MACCS;
- RDKit descriptors, where available.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path("../..").resolve()
RESULTS_ROOT = PROJECT_ROOT / "results" / "hi" / "sol"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RESULTS_ROOT: {RESULTS_ROOT}")
print(f"RESULTS_ROOT exists: {RESULTS_ROOT.exists()}")

PROJECT_ROOT: /home/f.capria/drug-discovery-lohi
RESULTS_ROOT: /home/f.capria/drug-discovery-lohi/results/hi/sol
RESULTS_ROOT exists: True


# Experiment registry

In [2]:
EXPERIMENTS = [
    # ---------------------------------------------------------------------
    # Decision Tree
    # ---------------------------------------------------------------------
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "dt_sol_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "dt_sol_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "dt_sol_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "dt_sol_hi_random_shuffle_maccs",
    },

    # ---------------------------------------------------------------------
    # Logistic Regression
    # ---------------------------------------------------------------------
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "lr_sol_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "lr_sol_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "OOD holdout",
        "result_dir": "lr_sol_hi_inner_ood_holdout_rdkit_desc",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "lr_sol_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "lr_sol_hi_random_shuffle_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "Random shuffle",
        "result_dir": "lr_sol_hi_random_shuffle_rdkit_desc",
    },

    # ---------------------------------------------------------------------
    # Linear SVM
    # ---------------------------------------------------------------------
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_sol_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_sol_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_sol_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_sol_hi_random_shuffle_maccs",
    },
]

registry = pd.DataFrame(EXPERIMENTS)
registry["path"] = registry["result_dir"].apply(lambda d: RESULTS_ROOT / d)
registry["exists"] = registry["path"].apply(lambda p: p.exists())

registry

,model,model_short,fingerprint,protocol,result_dir,path,exists
0,Decision Tree,DT,ECFP4,OOD holdout,dt_sol_hi_inner_ood_holdout_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/sol/dt_sol_hi_inner_ood_holdout_ecfp4,True
1,Decision Tree,DT,MACCS,OOD holdout,dt_sol_hi_inner_ood_holdout_maccs,/home/f.capria/drug-discovery-lohi/results/hi/sol/dt_sol_hi_inner_ood_holdout_maccs,True
2,Decision Tree,DT,ECFP4,Random shuffle,dt_sol_hi_random_shuffle_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/sol/dt_sol_hi_random_shuffle_ecfp4,True
3,Decision Tree,DT,MACCS,Random shuffle,dt_sol_hi_random_shuffle_maccs,/home/f.capria/drug-discovery-lohi/results/hi/sol/dt_sol_hi_random_shuffle_maccs,True
4,Logistic Regression,LR,ECFP4,OOD holdout,lr_sol_hi_inner_ood_holdout_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/sol/lr_sol_hi_inner_ood_holdout_ecfp4,True
5,Logistic Regression,LR,MACCS,OOD holdout,lr_sol_hi_inner_ood_holdout_maccs,/home/f.capria/drug-discovery-lohi/results/hi/sol/lr_sol_hi_inner_ood_holdout_maccs,True
6,Logistic Regression,LR,RDKit desc,OOD holdout,lr_sol_hi_inner_ood_holdout_rdkit_desc,/home/f.capria/drug-discovery-lohi/results/hi/sol/lr_sol_hi_inner_ood_holdout_rdkit_desc,True
7,Logistic Regression,LR,ECFP4,Random shuffle,lr_sol_hi_random_shuffle_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/sol/lr_sol_hi_random_shuffle_ecfp4,True
8,Logistic Regression,LR,MACCS,Random shuffle,lr_sol_hi_random_shuffle_maccs,/home/f.capria/drug-discovery-lohi/results/hi/sol/lr_sol_hi_random_shuffle_maccs,True
9,Logistic Regression,LR,RDKit desc,Random shuffle,lr_sol_hi_random_shuffle_rdkit_desc,/home/f.capria/drug-discovery-lohi/results/hi/sol/lr_sol_hi_random_shuffle_rdkit_desc,True


In [3]:
missing = registry.loc[~registry["exists"], ["model", "fingerprint", "protocol", "path"]]

if len(missing) > 0:
    print("Missing result folders:")
    display(missing)
else:
    print("All registered result folders exist.")

All registered result folders exist.


# Load params_fold_i.json into df_folds

In [4]:
def load_params_json(path: Path) -> dict:
    with open(path, "r") as f:
        return json.load(f)


def safe_get_metric(metrics: dict, key: str):
    if metrics is None:
        return np.nan
    return metrics.get(key, np.nan)


rows = []

for exp in EXPERIMENTS:
    result_path = RESULTS_ROOT / exp["result_dir"]

    for fold in [1, 2, 3]:
        params_path = result_path / f"params_fold_{fold}.json"

        if not params_path.exists():
            print(f"Missing file: {params_path}")
            continue

        data = load_params_json(params_path)

        train_metrics = data.get("train_metrics", {})
        test_metrics = data.get("test_metrics", {})
        best_params = data.get("best_params", {})

        inner_pr_auc = data.get("inner_selection_score", np.nan)
        inner_train_pr_auc = data.get("inner_train_score", np.nan)
        train_pr_auc = safe_get_metric(train_metrics, "pr_auc")
        test_pr_auc = safe_get_metric(test_metrics, "pr_auc")

        row = {
            "model": exp["model"],
            "model_short": exp["model_short"],
            "fingerprint": exp["fingerprint"],
            "protocol": exp["protocol"],
            "result_dir": exp["result_dir"],
            "fold": fold,

            "inner_pr_auc": inner_pr_auc,
            "inner_train_pr_auc": inner_train_pr_auc,
            "train_pr_auc": train_pr_auc,
            "test_pr_auc": test_pr_auc,

            "inner_test_gap": inner_pr_auc - test_pr_auc,
            "train_test_gap": train_pr_auc - test_pr_auc,

            "best_params": best_params,
            "inner_split_strategy": data.get("inner_split_strategy", None),
            "time_seconds": data.get("time_seconds", np.nan),
        }

        rows.append(row)

df_folds = pd.DataFrame(rows)

order_model = {
    "Decision Tree": 0,
    "Logistic Regression": 1,
    "Linear SVM": 2,
}

order_fp = {
    "ECFP4": 0,
    "MACCS": 1,
    "RDKit desc": 2,
}

order_protocol = {
    "OOD holdout": 0,
    "Random shuffle": 1,
}

df_folds["model_order"] = df_folds["model"].map(order_model)
df_folds["fingerprint_order"] = df_folds["fingerprint"].map(order_fp)
df_folds["protocol_order"] = df_folds["protocol"].map(order_protocol)

df_folds = df_folds.sort_values(
    ["model_order", "fingerprint_order", "protocol_order", "fold"]
).reset_index(drop=True)

print(f"Loaded rows: {len(df_folds)}")
df_folds.head(2)

Loaded rows: 42


,model,model_short,fingerprint,protocol,result_dir,fold,inner_pr_auc,inner_train_pr_auc,train_pr_auc,test_pr_auc,inner_test_gap,train_test_gap,best_params,inner_split_strategy,time_seconds,model_order,fingerprint_order,protocol_order
0,Decision Tree,DT,ECFP4,OOD holdout,dt_sol_hi_inner_ood_holdout_ecfp4,1,0.390689,0.589258,0.5752,0.3182,0.072489,0.2570,"{'ccp_alpha': 0.001, 'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 20, 'max_features': 'sqrt', 'm...",holdout,9.3,0,0,0
1,Decision Tree,DT,ECFP4,OOD holdout,dt_sol_hi_inner_ood_holdout_ecfp4,2,0.378160,0.657379,0.7060,0.2997,0.078460,0.4063,"{'ccp_alpha': 0.0, 'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 15, 'max_features': None, 'min_s...",holdout,4.5,0,0,0


# Per-fold table

In [5]:
per_fold_table = df_folds[
    [
        "model",
        "fingerprint",
        "protocol",
        "fold",
        "inner_pr_auc",
        "inner_train_pr_auc",
        "train_pr_auc",
        "test_pr_auc",
        "inner_test_gap",
        "train_test_gap",
    ]
].copy()

numeric_cols = [
    "inner_pr_auc",
    "inner_train_pr_auc",
    "train_pr_auc",                                                                                         
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

per_fold_table[numeric_cols] = per_fold_table[numeric_cols].round(4)

per_fold_table

,model,fingerprint,protocol,fold,inner_pr_auc,inner_train_pr_auc,train_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,Decision Tree,ECFP4,OOD holdout,1,0.3907,0.5893,0.5752,0.3182,0.0725,0.2570
1,Decision Tree,ECFP4,OOD holdout,2,0.3782,0.6574,0.7060,0.2997,0.0785,0.4063
2,Decision Tree,ECFP4,OOD holdout,3,0.3723,0.7987,0.7555,0.2934,0.0789,0.4621
3,Decision Tree,ECFP4,Random shuffle,1,0.5340,0.7958,0.8287,0.3107,0.2233,0.5180
4,Decision Tree,ECFP4,Random shuffle,2,0.4890,0.7020,0.8015,0.3009,0.1881,0.5006
5,Decision Tree,ECFP4,Random shuffle,3,0.4219,0.8736,0.8464,0.2806,0.1413,0.5658
6,Decision Tree,MACCS,OOD holdout,1,0.3969,0.6291,0.5089,0.3378,0.0591,0.1711
7,Decision Tree,MACCS,OOD holdout,2,0.3811,0.4939,0.4459,0.3459,0.0352,0.1000
8,Decision Tree,MACCS,OOD holdout,3,0.3905,0.6071,0.5326,0.3142,0.0763,0.2184
9,Decision Tree,MACCS,Random shuffle,1,0.4667,0.5617,0.5965,0.4016,0.0651,0.1949


# Aggregated summary table

In [6]:
def mean_std_string(mean_value, std_value, digits=4):
    if pd.isna(mean_value):
        return ""
    if pd.isna(std_value):
        return f"{mean_value:.{digits}f}"
    return f"{mean_value:.{digits}f} ± {std_value:.{digits}f}"


summary_numeric = (
    df_folds
    .groupby(["model", "model_short", "fingerprint", "protocol"], as_index=False)
    .agg(
        inner_mean=("inner_pr_auc", "mean"),
        inner_std=("inner_pr_auc", "std"),
        inner_train_mean=("inner_train_pr_auc", "mean"),
        inner_train_std=("inner_train_pr_auc", "std"),
        train_mean=("train_pr_auc", "mean"),
        train_std=("train_pr_auc", "std"),
        test_mean=("test_pr_auc", "mean"),
        test_std=("test_pr_auc", "std"),
        inner_test_gap_mean=("inner_test_gap", "mean"),
        inner_test_gap_std=("inner_test_gap", "std"),
        train_test_gap_mean=("train_test_gap", "mean"),
        train_test_gap_std=("train_test_gap", "std"),
    )
)

summary_numeric["model_order"] = summary_numeric["model"].map(order_model)
summary_numeric["fingerprint_order"] = summary_numeric["fingerprint"].map(order_fp)
summary_numeric["protocol_order"] = summary_numeric["protocol"].map(order_protocol)

summary_numeric = summary_numeric.sort_values(
    ["model_order", "fingerprint_order", "protocol_order"]
).reset_index(drop=True)

summary_table = summary_numeric[
    ["model", "fingerprint", "protocol"]
].copy()

summary_table["Inner PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["inner_mean"], r["inner_std"]), axis=1
)

summary_table["Inner train PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["inner_train_mean"], r["inner_train_std"]), axis=1
)

summary_table["Train PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["train_mean"], r["train_std"]), axis=1
)

summary_table["Final OOD test PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["test_mean"], r["test_std"]), axis=1
)

summary_table["Inner-test gap"] = summary_numeric.apply(
    lambda r: mean_std_string(r["inner_test_gap_mean"], r["inner_test_gap_std"]), axis=1
)

summary_table["Train-test gap"] = summary_numeric.apply(
    lambda r: mean_std_string(r["train_test_gap_mean"], r["train_test_gap_std"]), axis=1
)

summary_table

,model,fingerprint,protocol,Inner PR-AUC,Inner train PR-AUC,Train PR-AUC,Final OOD test PR-AUC,Inner-test gap,Train-test gap
0,Decision Tree,ECFP4,OOD holdout,0.3804 ± 0.0094,0.6818 ± 0.1068,0.6789 ± 0.0932,0.3038 ± 0.0129,0.0766 ± 0.0036,0.3751 ± 0.1060
1,Decision Tree,ECFP4,Random shuffle,0.4816 ± 0.0564,0.7905 ± 0.0859,0.8255 ± 0.0226,0.2974 ± 0.0154,0.1842 ± 0.0412,0.5281 ± 0.0338
2,Decision Tree,MACCS,OOD holdout,0.3895 ± 0.0079,0.5767 ± 0.0725,0.4958 ± 0.0448,0.3326 ± 0.0165,0.0569 ± 0.0206,0.1632 ± 0.0596
3,Decision Tree,MACCS,Random shuffle,0.4766 ± 0.0191,0.5744 ± 0.0198,0.6028 ± 0.0291,0.3553 ± 0.0470,0.1213 ± 0.0493,0.2475 ± 0.0457
4,Logistic Regression,ECFP4,OOD holdout,0.5024 ± 0.0345,0.8465 ± 0.1484,0.8304 ± 0.1247,0.4778 ± 0.0493,0.0247 ± 0.0833,0.3526 ± 0.1669
5,Logistic Regression,ECFP4,Random shuffle,0.5838 ± 0.0470,0.8953 ± 0.0321,0.8785 ± 0.0258,0.4820 ± 0.0481,0.1017 ± 0.0210,0.3964 ± 0.0631
6,Logistic Regression,MACCS,OOD holdout,0.4868 ± 0.0145,0.6964 ± 0.0954,0.6618 ± 0.0588,0.4768 ± 0.0210,0.0101 ± 0.0206,0.1850 ± 0.0515
7,Logistic Regression,MACCS,Random shuffle,0.5411 ± 0.0463,0.5789 ± 0.0851,0.5768 ± 0.0827,0.4612 ± 0.0332,0.0799 ± 0.0375,0.1157 ± 0.0903
8,Logistic Regression,RDKit desc,OOD holdout,0.5822 ± 0.0198,0.7890 ± 0.0379,0.7357 ± 0.0267,0.5908 ± 0.0406,-0.0086 ± 0.0279,0.1449 ± 0.0636
9,Logistic Regression,RDKit desc,Random shuffle,0.6433 ± 0.0380,0.7620 ± 0.0675,0.7511 ± 0.0604,0.5875 ± 0.0271,0.0558 ± 0.0405,0.1635 ± 0.0377


# Delta table

In [7]:
pivot_summary = summary_numeric.pivot_table(
    index=["model", "model_short", "fingerprint"],
    columns="protocol",
    values=[
        "inner_mean",
        "test_mean",
        "inner_test_gap_mean",
        "train_test_gap_mean",
    ],
)

delta_rows = []

for idx, row in pivot_summary.iterrows():
    model, model_short, fingerprint = idx

    try:
        random_inner = row[("inner_mean", "Random shuffle")]
        ood_inner = row[("inner_mean", "OOD holdout")]

        random_test = row[("test_mean", "Random shuffle")]
        ood_test = row[("test_mean", "OOD holdout")]

        random_inner_gap = row[("inner_test_gap_mean", "Random shuffle")]
        ood_inner_gap = row[("inner_test_gap_mean", "OOD holdout")]

        random_train_gap = row[("train_test_gap_mean", "Random shuffle")]
        ood_train_gap = row[("train_test_gap_mean", "OOD holdout")]

    except KeyError:
        continue

    delta_rows.append({
        "model": model,
        "model_short": model_short,
        "fingerprint": fingerprint,

        "ood_inner_mean": ood_inner,
        "random_inner_mean": random_inner,
        "delta_inner": random_inner - ood_inner,

        "ood_test_mean": ood_test,
        "random_test_mean": random_test,
        "delta_test": random_test - ood_test,

        "ood_inner_test_gap": ood_inner_gap,
        "random_inner_test_gap": random_inner_gap,
        "delta_inner_test_gap": random_inner_gap - ood_inner_gap,

        "ood_train_test_gap": ood_train_gap,
        "random_train_test_gap": random_train_gap,
        "delta_train_test_gap": random_train_gap - ood_train_gap,
    })

delta_table = pd.DataFrame(delta_rows)

delta_table["model_order"] = delta_table["model"].map(order_model)
delta_table["fingerprint_order"] = delta_table["fingerprint"].map(order_fp)

delta_table = delta_table.sort_values(
    ["model_order", "fingerprint_order"]
).reset_index(drop=True)

delta_display = delta_table[
    [
        "model",
        "fingerprint",
        "ood_inner_mean",
        "random_inner_mean",
        "delta_inner",
        "ood_test_mean",
        "random_test_mean",
        "delta_test",
        "ood_inner_test_gap",
        "random_inner_test_gap",
        "delta_inner_test_gap",
        "delta_train_test_gap",
    ]
].copy()

delta_numeric_cols = delta_display.select_dtypes(include=[np.number]).columns
delta_display[delta_numeric_cols] = delta_display[delta_numeric_cols].round(4)

delta_display

,model,fingerprint,ood_inner_mean,random_inner_mean,delta_inner,ood_test_mean,random_test_mean,delta_test,ood_inner_test_gap,random_inner_test_gap,delta_inner_test_gap,delta_train_test_gap
0,Decision Tree,ECFP4,0.3804,0.4816,0.1012,0.3038,0.2974,-0.0064,0.0766,0.1842,0.1076,0.1530
1,Decision Tree,MACCS,0.3895,0.4766,0.0871,0.3326,0.3553,0.0227,0.0569,0.1213,0.0645,0.0843
2,Logistic Regression,ECFP4,0.5024,0.5838,0.0813,0.4778,0.4820,0.0043,0.0247,0.1017,0.0770,0.0438
3,Logistic Regression,MACCS,0.4868,0.5411,0.0543,0.4768,0.4612,-0.0156,0.0101,0.0799,0.0699,-0.0694
4,Logistic Regression,RDKit desc,0.5822,0.6433,0.0611,0.5908,0.5875,-0.0032,-0.0086,0.0558,0.0643,0.0186
5,Linear SVM,ECFP4,0.4794,0.5886,0.1091,0.4968,0.4629,-0.0338,-0.0173,0.1256,0.1430,0.1070
6,Linear SVM,MACCS,0.4814,0.5405,0.0591,0.4756,0.4513,-0.0243,0.0058,0.0892,0.0834,-0.0316


# Hyperparameter summary tables

In [8]:
def flatten_best_params(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in df.iterrows():
        base = {
            "model": row["model"],
            "fingerprint": row["fingerprint"],
            "protocol": row["protocol"],
            "fold": row["fold"],
            "inner_pr_auc": row["inner_pr_auc"],
            "test_pr_auc": row["test_pr_auc"],
        }

        params = row["best_params"]

        if isinstance(params, dict):
            for key, value in params.items():
                clean_key = key.replace("model__", "")
                base[clean_key] = value

        rows.append(base)

    return pd.DataFrame(rows)


hp_df = flatten_best_params(df_folds)

hp_df["model_order"] = hp_df["model"].map(order_model)
hp_df["fingerprint_order"] = hp_df["fingerprint"].map(order_fp)
hp_df["protocol_order"] = hp_df["protocol"].map(order_protocol)

hp_df = hp_df.sort_values(
    ["model_order", "fingerprint_order", "protocol_order", "fold"]
).reset_index(drop=True)

for col in ["inner_pr_auc", "test_pr_auc"]:
    hp_df[col] = hp_df[col].round(4)

hp_df.head()

,model,fingerprint,protocol,fold,inner_pr_auc,test_pr_auc,ccp_alpha,class_weight,criterion,max_depth,max_features,min_samples_leaf,min_samples_split,C,l1_ratio,model_order,fingerprint_order,protocol_order
0,Decision Tree,ECFP4,OOD holdout,1,0.3907,0.3182,0.001,balanced,entropy,20.0,sqrt,5.0,20.0,NaN,NaN,0,0,0
1,Decision Tree,ECFP4,OOD holdout,2,0.3782,0.2997,0.000,balanced,entropy,15.0,NaN,10.0,2.0,NaN,NaN,0,0,0
2,Decision Tree,ECFP4,OOD holdout,3,0.3723,0.2934,0.001,NaN,gini,20.0,NaN,5.0,2.0,NaN,NaN,0,0,0
3,Decision Tree,ECFP4,Random shuffle,1,0.5340,0.3107,0.000,balanced,gini,15.0,NaN,5.0,2.0,NaN,NaN,0,0,1
4,Decision Tree,ECFP4,Random shuffle,2,0.4890,0.3009,0.000,NaN,entropy,15.0,sqrt,2.0,2.0,NaN,NaN,0,0,1


# Logistic Regression hyperparameters

In [9]:
lr_hp_table = hp_df.loc[
    hp_df["model"] == "Logistic Regression",
    [
        "protocol",
        "fingerprint",
        "fold",
        "C",
        "class_weight",
        "l1_ratio",
        "inner_pr_auc",
        "test_pr_auc",
    ]
].copy()

lr_hp_table = lr_hp_table.sort_values(
    ["fingerprint", "protocol", "fold"]
).reset_index(drop=True)

lr_hp_table

,protocol,fingerprint,fold,C,class_weight,l1_ratio,inner_pr_auc,test_pr_auc
0,OOD holdout,ECFP4,1,0.050,NaN,0.00,0.4757,0.5053
1,OOD holdout,ECFP4,2,0.100,balanced,0.50,0.4903,0.5072
2,OOD holdout,ECFP4,3,0.500,NaN,0.50,0.5414,0.4208
3,Random shuffle,ECFP4,1,0.100,balanced,0.00,0.5655,0.4876
4,Random shuffle,ECFP4,2,0.050,NaN,0.00,0.6371,0.5271
5,Random shuffle,ECFP4,3,0.500,NaN,1.00,0.5487,0.4314
6,OOD holdout,MACCS,1,0.100,NaN,0.25,0.4985,0.4671
7,OOD holdout,MACCS,2,10.000,NaN,0.50,0.4913,0.5009
8,OOD holdout,MACCS,3,0.500,NaN,0.00,0.4707,0.4623
9,Random shuffle,MACCS,1,0.005,balanced,0.00,0.4894,0.4312


# Svm liner hyperparameters

In [10]:
svm_hp_table = hp_df.loc[
    hp_df["model"] == "Linear SVM",
    [
        "protocol",
        "fingerprint",
        "fold",
        "C",
        "class_weight",
        "inner_pr_auc",
        "test_pr_auc",
    ]
].copy()

svm_hp_table = svm_hp_table.sort_values(
    ["fingerprint", "protocol", "fold"]
).reset_index(drop=True)

svm_hp_table

,protocol,fingerprint,fold,C,class_weight,inner_pr_auc,test_pr_auc
0,OOD holdout,ECFP4,1,0.010,balanced,0.4645,0.5082
1,OOD holdout,ECFP4,2,0.005,balanced,0.4529,0.5282
2,OOD holdout,ECFP4,3,0.010,balanced,0.5209,0.4539
3,Random shuffle,ECFP4,1,0.100,NaN,0.5601,0.3956
4,Random shuffle,ECFP4,2,0.010,balanced,0.6559,0.5394
5,Random shuffle,ECFP4,3,0.010,balanced,0.5497,0.4538
6,OOD holdout,MACCS,1,0.010,NaN,0.4870,0.4933
7,OOD holdout,MACCS,2,0.500,balanced,0.4957,0.4753
8,OOD holdout,MACCS,3,0.050,NaN,0.4614,0.4581
9,Random shuffle,MACCS,1,0.005,balanced,0.4990,0.4409


# Decision Tree hyperparameters

In [11]:
dt_hp_table = hp_df.loc[
    hp_df["model"] == "Decision Tree",
    [
        "protocol",
        "fingerprint",
        "fold",
        "criterion",
        "max_depth",
        "max_features",
        "min_samples_leaf",
        "min_samples_split",
        "ccp_alpha",
        "class_weight",
        "inner_pr_auc",
        "test_pr_auc",
    ]
].copy()

dt_hp_table = dt_hp_table.sort_values(
    ["fingerprint", "protocol", "fold"]
).reset_index(drop=True)

dt_hp_table

,protocol,fingerprint,fold,criterion,max_depth,max_features,min_samples_leaf,min_samples_split,ccp_alpha,class_weight,inner_pr_auc,test_pr_auc
0,OOD holdout,ECFP4,1,entropy,20.0,sqrt,5.0,20.0,0.0010,balanced,0.3907,0.3182
1,OOD holdout,ECFP4,2,entropy,15.0,NaN,10.0,2.0,0.0000,balanced,0.3782,0.2997
2,OOD holdout,ECFP4,3,gini,20.0,NaN,5.0,2.0,0.0010,NaN,0.3723,0.2934
3,Random shuffle,ECFP4,1,gini,15.0,NaN,5.0,2.0,0.0000,balanced,0.5340,0.3107
4,Random shuffle,ECFP4,2,entropy,15.0,sqrt,2.0,2.0,0.0000,NaN,0.4890,0.3009
5,Random shuffle,ECFP4,3,entropy,15.0,NaN,2.0,20.0,0.0000,NaN,0.4219,0.2806
6,OOD holdout,MACCS,1,gini,5.0,NaN,5.0,2.0,0.0010,NaN,0.3969,0.3378
7,OOD holdout,MACCS,2,gini,5.0,sqrt,2.0,2.0,0.0000,NaN,0.3811,0.3459
8,OOD holdout,MACCS,3,gini,10.0,log2,5.0,20.0,0.0000,balanced,0.3905,0.3142
9,Random shuffle,MACCS,1,gini,10.0,sqrt,5.0,20.0,0.0000,balanced,0.4667,0.4016


# Compact hyperparameter set summary

In [12]:
def unique_values_as_string(series: pd.Series) -> str:
    values = []
    for value in series.dropna().tolist():
        if value not in values:
            values.append(value)
    return "{" + ", ".join(str(v) for v in values) + "}"


hp_set_summary_rows = []

for model_name, model_df in hp_df.groupby("model"):
    param_cols = [
        c for c in model_df.columns
        if c not in [
            "model",
            "fingerprint",
            "protocol",
            "fold",
            "inner_pr_auc",
            "test_pr_auc",
            "model_order",
            "fingerprint_order",
            "protocol_order",
        ]
    ]

    for protocol, protocol_df in model_df.groupby("protocol"):
        row = {
            "model": model_name,
            "protocol": protocol,
        }

        for col in param_cols:
            if protocol_df[col].notna().any():
                row[col] = unique_values_as_string(protocol_df[col])

        hp_set_summary_rows.append(row)

hp_set_summary = pd.DataFrame(hp_set_summary_rows)
hp_set_summary["model_order"] = hp_set_summary["model"].map(order_model)
hp_set_summary["protocol_order"] = hp_set_summary["protocol"].map(order_protocol)

hp_set_summary = hp_set_summary.sort_values(
    ["model_order", "protocol_order"]
).drop(columns=["model_order", "protocol_order"]).reset_index(drop=True)

hp_set_summary

,model,protocol,ccp_alpha,class_weight,criterion,max_depth,max_features,min_samples_leaf,min_samples_split,C,l1_ratio
0,Decision Tree,OOD holdout,"{0.001, 0.0}",{balanced},"{entropy, gini}","{20.0, 15.0, 5.0, 10.0}","{sqrt, log2}","{5.0, 10.0, 2.0}","{20.0, 2.0}",NaN,NaN
1,Decision Tree,Random shuffle,"{0.0, 0.0001, 0.001}",{balanced},"{gini, entropy}","{15.0, 10.0, 20.0}","{sqrt, log2}","{5.0, 2.0, 1.0}","{2.0, 20.0}",NaN,NaN
2,Logistic Regression,OOD holdout,NaN,{balanced},NaN,NaN,NaN,NaN,NaN,"{0.05, 0.1, 0.5, 10.0}","{0.0, 0.5, 0.25}"
3,Logistic Regression,Random shuffle,NaN,{balanced},NaN,NaN,NaN,NaN,NaN,"{0.1, 0.05, 0.5, 0.005, 1.0}","{0.0, 1.0, 0.75, 0.5}"
4,Linear SVM,OOD holdout,NaN,{balanced},NaN,NaN,NaN,NaN,NaN,"{0.01, 0.005, 0.5, 0.05}",NaN
5,Linear SVM,Random shuffle,NaN,{balanced},NaN,NaN,NaN,NaN,NaN,"{0.1, 0.01, 0.005, 0.25}",NaN


# Save processed tables for the plotting notebook

In [13]:
TASK = "hi"
DATASET = "sol"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "results_ood_vs_random_shuffle"
    / TASK
    / DATASET
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Main protocol comparison tables
df_folds.to_csv(
    OUTPUT_DIR / "protocol_per_fold.csv",
    index=False,
)

summary_numeric.to_csv(
    OUTPUT_DIR / "protocol_summary_numeric.csv",
    index=False,
)

summary_table.to_csv(
    OUTPUT_DIR / "protocol_summary_display.csv",
    index=False,
)

delta_table.to_csv(
    OUTPUT_DIR / "protocol_delta.csv",
    index=False,
)

# Hyperparameter tables
hp_df.to_csv(
    OUTPUT_DIR / "hyperparameters_all.csv",
    index=False,
)

lr_hp_table.to_csv(
    OUTPUT_DIR / "hyperparameters_lr.csv",
    index=False,
)

svm_hp_table.to_csv(
    OUTPUT_DIR / "hyperparameters_svm.csv",
    index=False,
)

dt_hp_table.to_csv(
    OUTPUT_DIR / "hyperparameters_dt.csv",
    index=False,
)

hp_set_summary.to_csv(
    OUTPUT_DIR / "hyperparameters_set_summary.csv",
    index=False,
)

print("Saved processed files in:")
print(OUTPUT_DIR)

print("\nFiles saved:")
for file in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"- {file.name}")

Saved processed files in:
/home/f.capria/drug-discovery-lohi/results/results_ood_vs_random_shuffle/hi/sol

Files saved:
- hyperparameters_all.csv
- hyperparameters_dt.csv
- hyperparameters_lr.csv
- hyperparameters_set_summary.csv
- hyperparameters_svm.csv
- protocol_delta.csv
- protocol_per_fold.csv
- protocol_summary_display.csv
- protocol_summary_numeric.csv
